In [ ]:
import numpy
import pandas
import cv2 as cv
from cv2.typing import MatLike
import matplotlib.pyplot as plt
from pathlib import Path
import sys

root: Path = Path.cwd()
if not (root / "src" / "ccr").is_dir():
    root: Path = root.parents[1]

src: Path = root / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

import ccr.labels as label

In [ ]:
def importImage(path:str):
   img: MatLike | None = cv.imread(path, cv.IMREAD_GRAYSCALE)
   plt.imshow(X=img, cmap="gray") # pyright: ignore[reportArgumentType]
   plt.show()
   return img


In [ ]:
sheets: list[Path] = label.discover_label_files(datasets_dir=root / "datasets")
files: list[Any] = []

for path in sheets:
    if path.parent.name == "synthetic":
        files.append(path)

data:list[label.Label] = label.load_labels(files.pop())

print (data)

In [ ]:
img: MatLike=importImage(path=str(data.iloc[4].image_path))
print (img)

In [ ]:
px1_avg:int= -1
px2:int= -1
px3_avg:int= -1

#locations where pixel ratio is if a pixel is white and 5 pixels before and behind are black
ratio_location: list[Any] = []
px3_state_values: list[5] = []
px1_state_values: list[5] = []

#The idea is to scan the part of the image where we expect the code to be
#We then average the value of the 5 pixels before and the value of the 5 pixels after to look for our pattern in the code
#At the moment this is not the most efficent but gives us conformation that the system can recognize this pattern
#The next step is to plot over these coordinates to see if they truly land on our expected code and aren't accidentally finding the pattern elsewhere
#If they are finding the pattern elsewhere then this can be scalled to look further on either side

print( img.shape)
for x in range (2500, img.shape[0]-5):
    for y in range (1000, img.shape[1]-1):
            px2 = img[x,y]
            if (x < img.shape[0] - 5 and x > 5):
                for i in range (1,6):
                    try:
                        px3_state_values.append(img[x+(3+i),y])
                        px1_state_values.append(img[x-(2+i),y])
                    except IndexError:
                        pass
                if (len(px3_state_values) > 0 and len(px1_state_values) > 0):
                    px3_avg = sum(int(item) for item in px3_state_values) / len(px3_state_values)
                    px1_avg = sum(int (item) for item in px1_state_values) / len(px1_state_values)
                    px3_state_values.clear()
                    px1_state_values.clear()       
            if (px1_avg==0 and px2==255 and px3_avg==0):
                ratio_location.append((x,y))


if ratio_location.__len__() > 0:
    print (ratio_location)
else:
    print ("No locations found")